In [ ]:
!git clone https://github.com/nirajj12/Bird-s-Eye-View-BEV-2D-Occupancy.git

fatal: destination path 'Bird-s-Eye-View-BEV-2D-Occupancy' already exists and is not an empty directory.


In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'numpy==1.26.4', '--force-reinstall', '-q'], check=True)
subprocess.run(['pip', 'install', 'scipy==1.11.4', '--force-reinstall', '-q'], check=True)
subprocess.run(['pip', 'install', 'scikit-learn==1.3.2', '--force-reinstall', '-q'], check=True)
print("Done!")

Done!


In [ ]:
!pip install -q nuscenes-devkit==1.1.11 --no-deps
!pip install -q pyquaternion descartes fire tqdm einops timm Pillow matplotlib


In [ ]:
import numpy, torch
print("numpy :", numpy.__version__)
print("torch :", torch.__version__)
print("GPU   :", torch.cuda.is_available())

from nuscenes.nuscenes import NuScenes
print("nuscenes OK ✓")

numpy : 1.26.4
torch : 2.10.0+cu128
GPU   : True
nuscenes OK ✓


In [ ]:
# Cell 1 — Remount Drive forcing account selection
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
# → A popup will appear asking WHICH Google account to use
# → Select the SAME account you see in the Drive screenshot above

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
# Cell 2 — Verify the file is now visible
import os
print(os.listdir('/content/drive/MyDrive'))
# Must show 'v1.0-mini.tgz' in the list

['1.png', '1653128258982 (1)_copy_400x534.jpg', 'DSC_0700.JPG', 'Colab Notebooks', 'Screenshot 2025-06-18 at 6.50.09\u202fPM.png', 'IBMDesign20250723-32-gobhrq.pdf', 'WhatsApp Image 2025-07-23 at 18.48.25.jpeg', 'Salary Project template.pptx', 'Free Internship Toolkit', 'master placement drive', 'Document ', 'Videos', 'NirajResume (1).pdf', 'NirajjResume.pdf', 'NirajjResume (1) (1).pdf', 'NirajjResume (1).pdf', 'NirajResume.pdf', 'Untitled Diagram.drawio', 'Screenshot 2026-02-23 at 9.10.05\u202fPM.png', 'v1 (1).0-mini.tgz', 'v1.0-mini.tgz', 'BEV_PROJECT', 'BEV_PROJECTss']


In [ ]:
# Cell 3 — Extract once confirmed visible
os.makedirs('/content/dataset', exist_ok=True)
!tar -xzf "/content/drive/MyDrive/v1.0-mini.tgz" -C /content/dataset/
print("Dataset:", os.listdir('/content/dataset'))
# Must show: ['maps', 'sweeps', 'samples', 'v1.0-mini', 'LICENSE']


Dataset: ['sweeps', '.v1.0-mini.txt', 'maps', 'v1.0-mini', 'LICENSE', 'samples']


In [ ]:

# Cell 1 — Clone repo + set paths
import os, sys

os.chdir('/content')
!git clone https://github.com/nirajj12/Bird-s-Eye-View-BEV-2D-Occupancy.git
os.chdir('/content/Bird-s-Eye-View-BEV-2D-Occupancy')
sys.path.insert(0, '/content/Bird-s-Eye-View-BEV-2D-Occupancy')

print("Files:", os.listdir('.'))

In [ ]:
# Run this FIRST — before any import
import os, sys

REPO = '/content/Bird-s-Eye-View-BEV-2D-Occupancy'

# ── Clone if not exists ───────────────────────────────────────────
if not os.path.exists(REPO):
    os.system(
        'git clone https://github.com/nirajj12/Bird-s-Eye-View-BEV-2D-Occupancy.git '
        + REPO
    )
    print("✅ Repo cloned")
else:
    os.system(f'cd {REPO} && git pull')
    print("✅ Repo updated")

# ── MUST be before ANY project import ────────────────────────────
os.chdir(REPO)
sys.path.insert(0, REPO)        # ← this was missing

print(f"CWD : {os.getcwd()}")
print(f"Path: {sys.path[0]}")

# ── Now safe to import ────────────────────────────────────────────
import config.config as cfg
cfg.DATAROOT = '/content/dataset'
print(f"✅ config loaded | DATAROOT: {cfg.DATAROOT}")

✅ Repo updated
CWD : /content/Bird-s-Eye-View-BEV-2D-Occupancy
Path: /content/Bird-s-Eye-View-BEV-2D-Occupancy
✅ config loaded | DATAROOT: /content/dataset


In [ ]:
# Cell 3 — Verify full pipeline on GPU
import torch
from models.bev_model import BEVOccupancyModel
from data.nuscenes_loader import get_dataloaders

device = torch.device('cuda')
print(f"GPU: {torch.cuda.get_device_name(0)}")

train_loader, val_loader, ts, vs = get_dataloaders(dataroot='/content/dataset')
print(f"Train: {ts} | Val: {vs}")

model = BEVOccupancyModel(pretrained=True).to(device)
batch = next(iter(train_loader))

with torch.no_grad():
    occ_logits, aux_logits = model(
        batch['imgs'].to(device),
        batch['intrinsics'].to(device),
        batch['extrinsics'].to(device)
    )
    losses = model.compute_loss(occ_logits, aux_logits, batch['occ_gt'].to(device))

print(f"Output : {tuple(occ_logits.shape)}")
print(f"Loss   : {losses['total'].item():.4f}")
print("Pipeline OK ✓ — safe to train")

GPU: Tesla T4


2026-03-24 17:25:08 | INFO     | data.nuscenes_loader | Dataset loaded | version: v1.0-mini | samples: 404
2026-03-24 17:25:08 | INFO     | data.nuscenes_loader | DataLoaders ready | train: 323 | val: 81 | pin_memory: True


Train: <torch.utils.data.dataset.Subset object at 0x7fa215f60200> | Val: <data.nuscenes_loader.BEVOccupancyDataset object at 0x7fa21cc50ef0>
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 197MB/s]
2026-03-24 17:25:10 | INFO     | models.backbone | ImageBackbone initialized | out_channels: 128 | pretrained: True
2026-03-24 17:25:10 | INFO     | models.bev_former_lite | BEVFormerLite V3.1 | BEV=200×200 | Z heights: [0.0, 0.5, 1.5] | N_pts/height=40000 | C=128
2026-03-24 17:25:10 | INFO     | models.bev_decoder | BEVDecoder initialized | 128 → 64 channels
2026-03-24 17:25:10 | INFO     | models.bev_decoder | OccupancyHead initialized | input: 64ch → output: 1ch logits
2026-03-24 17:25:10 | INFO     | models.bev_model | BEVOccupancyModel ready | total: 9,357,890 | trainable: 9,357,890


Output : (2, 1, 200, 200)
Loss   : 1.2476
Pipeline OK ✓ — safe to train


In [ ]:
# ══════════════════════════════════════════════════════════════════
# V3 FINAL TRAINING — BEV 2D Occupancy
# Fixed: clone guard, DWE resume, dual plateau tracking
# ══════════════════════════════════════════════════════════════════

import os, sys, torch, torch.optim as optim
import numpy as np
from tqdm import tqdm

# ── Mount Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Clone only if not already cloned ──────────────────────────────
REPO = '/content/Bird-s-Eye-View-BEV-2D-Occupancy'
if not os.path.exists(REPO):
    os.system(
        'git clone https://github.com/nirajj12/Bird-s-Eye-View-BEV-2D-Occupancy.git '
        '/content/Bird-s-Eye-View-BEV-2D-Occupancy'
    )
    print("✅ Repo cloned")
else:
    os.system(f'cd {REPO} && git pull')   # pull latest changes
    print("✅ Repo pulled latest")

os.chdir(REPO)
sys.path.insert(0, REPO)
os.system('pip install nuscenes-devkit pyquaternion -q')

# ── Config override ───────────────────────────────────────────────
import config.config as cfg
cfg.DATAROOT = '/content/dataset'

from data.nuscenes_loader import get_dataloaders
from models.bev_model     import BEVOccupancyModel
from utils.metrics        import compute_metrics

# ── Device ────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {torch.cuda.get_device_name(0)}")

# ── Drive paths ───────────────────────────────────────────────────
DRIVE_SAVE = '/content/drive/MyDrive/BEV_PROJECT_V3'
os.makedirs(DRIVE_SAVE, exist_ok=True)

# ── Data ──────────────────────────────────────────────────────────
train_loader, val_loader, val_ds, full_dataset = get_dataloaders(
    dataroot='/content/dataset'
)
print(f"Train  : {len(train_loader.dataset)}")
print(f"Val    : {len(val_ds)}")

# ── Hyperparams ───────────────────────────────────────────────────
TOTAL_EP      = 60
WARMUP_EP     = 5
PATIENCE_IoU  = 12   # stop if smooth IoU flat for 12 epochs
PATIENCE_DWE  = 15   # stop if DWE flat for 15 epochs (longer — DWE improves slower)

# ── Model + Optimizer ─────────────────────────────────────────────
model     = BEVOccupancyModel(pretrained=True).to(device)
optimizer = optim.AdamW(
    model.parameters(), lr=2e-4, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=55, eta_min=1e-6
)

# ── State trackers ────────────────────────────────────────────────
best_iou        = 0.0
best_dwe        = float('inf')
best_smooth_iou = 0.0
best_smooth_dwe = float('inf')
history         = []
start_epoch     = 1
no_improve_iou  = 0
no_improve_dwe  = 0

# ── Auto-resume ───────────────────────────────────────────────────
ckpt_files = sorted([
    f for f in os.listdir(DRIVE_SAVE)
    if f.startswith('epoch_') and f.endswith('.pth')
])

if ckpt_files:
    latest_path = os.path.join(DRIVE_SAVE, ckpt_files[-1])
    print(f"\n🔄 Resuming from : {ckpt_files[-1]}")

    ckpt        = torch.load(latest_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    history     = ckpt.get('history', [])

    if history:
        best_iou        = max(h['val_iou'] for h in history)
        best_dwe        = min(h['val_dwe'] for h in history)  # ✅ FIX: was missing
        best_smooth_iou = best_iou
        best_smooth_dwe = best_dwe
    else:
        best_iou = ckpt['val_iou']
        best_dwe = ckpt['val_dwe']

    # Restore scheduler state
    for _ in range(max(0, ckpt['epoch'] - WARMUP_EP)):
        scheduler.step()

    print(f"✅ Start epoch  : {start_epoch}")
    print(f"✅ Best IoU     : {best_iou:.4f}")
    print(f"✅ Best DWE     : {best_dwe:.4f}")
else:
    print("\n🆕 No checkpoint — starting fresh from epoch 1")

print(f"📁 Saving to    : {DRIVE_SAVE}")
print(f"📊 Epochs       : {start_epoch} → {TOTAL_EP}")
print("─" * 85)
print(f"{'Ep':>4} | {'Loss':>7} | {'Focal':>6} | {'Dice':>6} | "
      f"{'DWELoss':>7} | {'Phase':>6} | {'IoU':>7} | {'DWE':>7} | Note")
print("─" * 85)

# ══════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ══════════════════════════════════════════════════════════════════
for epoch in range(start_epoch, TOTAL_EP + 1):

    # ── Warmup LR schedule ────────────────────────────────────────
    if epoch <= WARMUP_EP:
        for pg in optimizer.param_groups:
            pg['lr'] = 2e-4 * epoch / WARMUP_EP

    # ────────────────────────────────────────────────────────────
    # TRAIN
    # ────────────────────────────────────────────────────────────
    model.train()
    ep_loss, ep_focal, ep_dice, ep_dwe_l, n_b = 0, 0, 0, 0, 0

    pbar = tqdm(
        train_loader,
        desc  = f"Ep {epoch:02d}/{TOTAL_EP} [Train]",
        ncols = 120
    )

    for batch in pbar:
        imgs       = batch['imgs'].to(device)
        intrinsics = batch['intrinsics'].to(device)
        extrinsics = batch['extrinsics'].to(device)
        occ_gt     = batch['occ_gt'].to(device)

        occ_logits, aux_logits = model(imgs, intrinsics, extrinsics)

        # ✅ epoch passed → warmup / phase1 / phase2 DWE
        losses = model.compute_loss(
            occ_logits, aux_logits, occ_gt,
            epoch = epoch
        )
        loss = losses['total']

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=2.0
        )
        optimizer.step()

        ep_loss  += loss.item()
        ep_focal += losses['focal'].item() if hasattr(losses['focal'], 'item') else losses['focal']
        ep_dice  += losses['dice'].item()  if hasattr(losses['dice'],  'item') else losses['dice']
        ep_dwe_l += losses['dwe'].item()   if hasattr(losses['dwe'],   'item') else losses['dwe']
        n_b      += 1

        pbar.set_postfix({
            'L'  : f"{loss.item():.3f}",
            'F'  : f"{losses['focal']:.3f}",
            'D'  : f"{losses['dice']:.3f}",
            'DWE': f"{losses['dwe']:.4f}",
            'Ph' : losses['phase'][:4],
        })

    avg_loss  = ep_loss  / n_b
    avg_focal = ep_focal / n_b
    avg_dice  = ep_dice  / n_b
    avg_dwe_l = ep_dwe_l / n_b

    if epoch > WARMUP_EP:
        scheduler.step()

    # ────────────────────────────────────────────────────────────
    # VALIDATE — per-sample (accurate DWE)
    # ────────────────────────────────────────────────────────────
    model.eval()
    iou_sum, dwe_sum, v_b = 0, 0, 0

    with torch.no_grad():
        for idx in val_ds.indices:
            s    = full_dataset[idx]
            imgs = s['imgs'].unsqueeze(0).to(device)
            K    = s['intrinsics'].unsqueeze(0).to(device)
            E    = s['extrinsics'].unsqueeze(0).to(device)
            gt   = s['occ_gt'].to(device)

            occ_logits, _ = model(imgs, K, E)
            m        = compute_metrics(occ_logits, gt.unsqueeze(0))
            iou_sum += m['occ_iou']
            dwe_sum += m['dwe']
            v_b     += 1

    avg_iou = iou_sum / v_b
    avg_dwe = dwe_sum / v_b

    history.append({
        'epoch'   : epoch,
        'loss'    : avg_loss,
        'focal'   : avg_focal,
        'dice'    : avg_dice,
        'dwe_loss': avg_dwe_l,
        'val_iou' : avg_iou,
        'val_dwe' : avg_dwe,
        'phase'   : losses['phase']
    })

    # ── Rolling averages (last 3 epochs) ─────────────────────────
    recent_ious = [h['val_iou'] for h in history[-3:]]
    recent_dwes = [h['val_dwe'] for h in history[-3:]]
    smooth_iou  = sum(recent_ious) / len(recent_ious)
    smooth_dwe  = sum(recent_dwes) / len(recent_dwes)

    # ────────────────────────────────────────────────────────────
    # SAVE BEST IoU MODEL
    # ────────────────────────────────────────────────────────────
    note = ""
    if avg_iou > best_iou:
        best_iou = avg_iou
        note    += " ★IoU"
        torch.save({
            'epoch'      : epoch,
            'model_state': model.state_dict(),
            'optimizer'  : optimizer.state_dict(),
            'val_iou'    : avg_iou,
            'val_dwe'    : avg_dwe,
            'history'    : history
        }, f'{DRIVE_SAVE}/best_iou_model.pth')

    # ── Save best DWE model (main target) ────────────────────────
    if avg_dwe < best_dwe:
        best_dwe = avg_dwe
        note    += " ★DWE"
        torch.save({
            'epoch'      : epoch,
            'model_state': model.state_dict(),
            'optimizer'  : optimizer.state_dict(),
            'val_iou'    : avg_iou,
            'val_dwe'    : avg_dwe,
            'history'    : history
        }, f'{DRIVE_SAVE}/best_dwe_model.pth')

    # ────────────────────────────────────────────────────────────
    # PLATEAU DETECTION (dual — IoU + DWE)
    # ────────────────────────────────────────────────────────────
    if smooth_iou > best_smooth_iou:
        best_smooth_iou = smooth_iou
        no_improve_iou  = 0
    else:
        no_improve_iou += 1

    if smooth_dwe < best_smooth_dwe:
        best_smooth_dwe = smooth_dwe
        no_improve_dwe  = 0
    else:
        no_improve_dwe += 1

    # ────────────────────────────────────────────────────────────
    # EPOCH LOG (table row)
    # ────────────────────────────────────────────────────────────
    print(f"{epoch:>4} | {avg_loss:>7.4f} | {avg_focal:>6.3f} | "
          f"{avg_dice:>6.3f} | {avg_dwe_l:>7.4f} | "
          f"{losses['phase']:>6} | {avg_iou:>7.4f} | "
          f"{avg_dwe:>7.4f} |{note} "
          f"iou_pat={no_improve_iou}/{PATIENCE_IoU} "
          f"dwe_pat={no_improve_dwe}/{PATIENCE_DWE}")

    # ────────────────────────────────────────────────────────────
    # EPOCH CHECKPOINT (every epoch — safe for Colab crashes)
    # ────────────────────────────────────────────────────────────
    torch.save({
        'epoch'      : epoch,
        'model_state': model.state_dict(),
        'optimizer'  : optimizer.state_dict(),
        'val_iou'    : avg_iou,
        'val_dwe'    : avg_dwe,
        'history'    : history
    }, f'{DRIVE_SAVE}/epoch_{epoch:02d}.pth')

    # ────────────────────────────────────────────────────────────
    # EARLY STOP — both IoU AND DWE must plateau
    # ────────────────────────────────────────────────────────────
    iou_plateau = no_improve_iou >= PATIENCE_IoU
    dwe_plateau = no_improve_dwe >= PATIENCE_DWE

    if iou_plateau and dwe_plateau:
        print(f"\n⏹  EARLY STOP")
        print(f"   IoU flat for {no_improve_iou} epochs "
              f"(patience={PATIENCE_IoU})")
        print(f"   DWE flat for {no_improve_dwe} epochs "
              f"(patience={PATIENCE_DWE})")
        break

    if iou_plateau and not dwe_plateau:
        print(f"   ⚠️  IoU plateau but DWE still improving "
              f"— continuing for DWE")

# ════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════════════
print(f"""
╔══════════════════════════════════════════════╗
║          V3 TRAINING COMPLETE               ║
╠══════════════════════════════════════════════╣
║  Best IoU : {best_iou:.4f}  (V2: 0.3011)         ║
║  Best DWE : {best_dwe:.4f}  (V2: 0.2323)         ║
╠══════════════════════════════════════════════╣
║  Δ IoU    : {best_iou - 0.3011:+.4f}                      ║
║  Δ DWE    : {best_dwe - 0.2323:+.4f}  ← main target      ║
╠══════════════════════════════════════════════╣
║  Checkpoints saved to BEV_PROJECT_V3/       ║
║    best_iou_model.pth  ← highest IoU        ║
║    best_dwe_model.pth  ← lowest DWE         ║
║    epoch_XX.pth        ← every epoch        ║
╚══════════════════════════════════════════════╝
""")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Repo pulled latest
Device : Tesla T4


2026-03-24 17:27:18 | INFO     | data.nuscenes_loader | Dataset loaded | version: v1.0-mini | samples: 404
2026-03-24 17:27:18 | INFO     | data.nuscenes_loader | DataLoaders ready | train: 323 | val: 81 | pin_memory: True


Train  : 323
Val    : 81


2026-03-24 17:27:19 | INFO     | models.backbone | ImageBackbone initialized | out_channels: 128 | pretrained: True
2026-03-24 17:27:19 | INFO     | models.bev_former_lite | BEVFormerLite V3.1 | BEV=200×200 | Z heights: [0.0, 0.5, 1.5] | N_pts/height=40000 | C=128
2026-03-24 17:27:19 | INFO     | models.bev_decoder | BEVDecoder initialized | 128 → 64 channels
2026-03-24 17:27:19 | INFO     | models.bev_decoder | OccupancyHead initialized | input: 64ch → output: 1ch logits
2026-03-24 17:27:19 | INFO     | models.bev_model | BEVOccupancyModel ready | total: 9,357,890 | trainable: 9,357,890



🆕 No checkpoint — starting fresh from epoch 1
📁 Saving to    : /content/drive/MyDrive/BEV_PROJECT_V3
📊 Epochs       : 1 → 60
─────────────────────────────────────────────────────────────────────────────────────
  Ep |    Loss |  Focal |   Dice | DWELoss |  Phase |     IoU |     DWE | Note
─────────────────────────────────────────────────────────────────────────────────────


...

  58 |  0.5921 |  0.118 |  0.347 |  0.0334 | phase2-DWE |  0.3608 |  0.0001 | iou_pat=2/12 dwe_pat=5/15


Ep 59/60 [Train]: 100%|███████████████| 162/162 [01:18<00:00,  2.07it/s, L=0.581, F=0.109, D=0.338, DWE=0.0362, Ph=phas]


  59 |  0.5910 |  0.118 |  0.346 |  0.0333 | phase2-DWE |  0.3616 |  0.0001 | ★DWE iou_pat=3/12 dwe_pat=0/15


Ep 60/60 [Train]: 100%|███████████████| 162/162 [01:18<00:00,  2.06it/s, L=0.579, F=0.111, D=0.347, DWE=0.0313, Ph=phas]


  60 |  0.5902 |  0.118 |  0.345 |  0.0333 | phase2-DWE |  0.3649 |  0.0001 | ★IoU iou_pat=4/12 dwe_pat=0/15

╔══════════════════════════════════════════════╗
║          V3 TRAINING COMPLETE               ║
╠══════════════════════════════════════════════╣
║  Best IoU : 0.3649  (V2: 0.3011)         ║
║  Best DWE : 0.0001  (V2: 0.2323)         ║
╠══════════════════════════════════════════════╣
║  Δ IoU    : +0.0638                      ║
║  Δ DWE    : -0.2322  ← main target      ║
╠══════════════════════════════════════════════╣
║  Checkpoints saved to BEV_PROJECT_V3/       ║
║    best_iou_model.pth  ← highest IoU        ║
║    best_dwe_model.pth  ← lowest DWE         ║
║    epoch_XX.pth        ← every epoch        ║
╚══════════════════════════════════════════════╝



In [ ]:
import torch, numpy as np

# Load V3 best IoU checkpoint
ckpt = torch.load('/content/drive/MyDrive/BEV_PROJECT_V3/best_iou_model.pth',
                  map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

THRESH = 0.5

def corrected_dwe(logits, gt_tensor):
    prob  = torch.sigmoid(logits)
    pred_b = (prob > THRESH).float()
    gt_b   = (gt_tensor > 0.5).float()
    H, W   = gt_b.shape[-2], gt_b.shape[-1]
    cx, cy = W // 2, H // 2
    ys = torch.arange(H, device=gt_b.device).float()
    xs = torch.arange(W, device=gt_b.device).float()
    yy, xx = torch.meshgrid(ys, xs, indexing='ij')
    dist = torch.sqrt((xx - cx)**2 + (yy - cy)**2).clamp(min=1.0)
    sp_w = (1.0 / dist)
    sp_w = sp_w / sp_w.sum()           # normalize
    error = torch.abs(pred_b - gt_b)
    return (sp_w * error.squeeze()).sum().item()

results = []
with torch.no_grad():
    for idx in val_ds.indices:
        sample = full_dataset[idx]
        imgs = sample['imgs'].unsqueeze(0).to(device)
        K    = sample['intrinsics'].unsqueeze(0).to(device)
        E    = sample['extrinsics'].unsqueeze(0).to(device)
        gt   = sample['occ_gt'].to(device)
        occ_logits, _ = model(imgs, K, E)
        dwe = corrected_dwe(occ_logits, gt.unsqueeze(0).unsqueeze(0))
        prob = torch.sigmoid(occ_logits.squeeze())
        pred = (prob > THRESH).float()
        gt_b = (gt > 0.5).float()
        tp = (pred * gt_b).sum().item()
        fp = (pred * (1 - gt_b)).sum().item()
        fn = ((1 - pred) * gt_b).sum().item()
        iou = tp / (tp + fp + fn + 1e-6)
        results.append({'iou': iou, 'dwe': dwe})

mean_iou = np.mean([r['iou'] for r in results])
mean_dwe = np.mean([r['dwe'] for r in results])
print(f"V3 CORRECTED → IoU: {mean_iou:.4f} | DWE: {mean_dwe:.6f}")
print(f"V2 baseline  → IoU: 0.3011       | DWE: 0.2323")
print(f"Delta IoU    → {mean_iou - 0.3011:+.4f}")
print(f"Delta DWE    → {mean_dwe - 0.2323:+.6f}")

V3 CORRECTED → IoU: 0.3649 | DWE: 0.113690
V2 baseline  → IoU: 0.3011       | DWE: 0.2323
Delta IoU    → +0.0638
Delta DWE    → -0.118610


In [ ]:
import os, json, torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from torch.utils.data import random_split
from data.nuscenes_loader import BEVOccupancyDataset

device = torch.device('cuda')
SAVE_DIR = '/content/drive/MyDrive/BEV_PROJECT_V3'
VIZ_DIR  = '/content/eval_viz'
os.makedirs(VIZ_DIR, exist_ok=True)

# Load model
from models.bev_model import BEVOccupancyModel
import config.config as cfg
cfg.DATAROOT = '/content/dataset'

model = BEVOccupancyModel(pretrained=False).to(device)
ckpt  = torch.load(f'{SAVE_DIR}/best_iou_model.pth', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded checkpoint → Epoch {ckpt['epoch']} | IoU {ckpt['val_iou']:.4f}")

# Reconstruct val split (same seed as training)
full_dataset = BEVOccupancyDataset(dataroot='/content/dataset')
total      = len(full_dataset)
train_size = int(0.8 * total)
val_size   = total - train_size
_, val_ds  = random_split(full_dataset, [train_size, val_size],
                           generator=torch.Generator().manual_seed(42))
print(f"Val samples: {len(val_ds)}")

2026-03-24 19:10:09 | INFO     | models.backbone | ImageBackbone initialized | out_channels: 128 | pretrained: False
2026-03-24 19:10:09 | INFO     | models.bev_former_lite | BEVFormerLite V3.1 | BEV=200×200 | Z heights: [0.0, 0.5, 1.5] | N_pts/height=40000 | C=128
2026-03-24 19:10:09 | INFO     | models.bev_decoder | BEVDecoder initialized | 128 → 64 channels
2026-03-24 19:10:09 | INFO     | models.bev_decoder | OccupancyHead initialized | input: 64ch → output: 1ch logits
2026-03-24 19:10:09 | INFO     | models.bev_model | BEVOccupancyModel ready | total: 9,357,890 | trainable: 9,357,890


Loaded checkpoint → Epoch 60 | IoU 0.3649


2026-03-24 19:10:10 | INFO     | data.nuscenes_loader | Dataset loaded | version: v1.0-mini | samples: 404


Val samples: 81


In [ ]:
THRESH = 0.5

def corrected_dwe(logits, gt_tensor):
    """Hard binary DWE — correct formulation"""
    prob   = torch.sigmoid(logits)
    pred_b = (prob > THRESH).float()
    gt_b   = (gt_tensor > 0.5).float()
    H, W   = gt_b.shape[-2], gt_b.shape[-1]
    cx, cy = W // 2, H // 2
    ys = torch.arange(H, device=gt_b.device).float()
    xs = torch.arange(W, device=gt_b.device).float()
    yy, xx = torch.meshgrid(ys, xs, indexing='ij')
    dist = torch.sqrt((xx - cx)**2 + (yy - cy)**2).clamp(min=1.0)
    sp_w = 1.0 / dist
    sp_w = sp_w / sp_w.sum()
    error = torch.abs(pred_b - gt_b)
    return (sp_w * error.squeeze()).sum().item()

def near_far_iou(pred_b, gt_b, radius=30):
    """Split IoU into near-ego vs far-field"""
    H, W   = gt_b.shape[-2], gt_b.shape[-1]
    cx, cy = W // 2, H // 2
    ys = torch.arange(H, device=gt_b.device).float()
    xs = torch.arange(W, device=gt_b.device).float()
    yy, xx = torch.meshgrid(ys, xs, indexing='ij')
    dist = torch.sqrt((xx - cx)**2 + (yy - cy)**2)
    near = (dist <= radius)
    far  = (dist >  radius)

    def iou_mask(mask):
        p = pred_b.squeeze()[mask]
        g = gt_b.squeeze()[mask]
        tp = (p * g).sum().item()
        fp = (p * (1 - g)).sum().item()
        fn = ((1 - p) * g).sum().item()
        return tp / (tp + fp + fn + 1e-6)

    return iou_mask(near), iou_mask(far)

results = []
with torch.no_grad():
    for i, idx in enumerate(val_ds.indices):
        sample = full_dataset[idx]
        imgs = sample['imgs'].unsqueeze(0).to(device)
        K    = sample['intrinsics'].unsqueeze(0).to(device)
        E    = sample['extrinsics'].unsqueeze(0).to(device)
        gt   = sample['occ_gt'].to(device)

        occ_logits, _ = model(imgs, K, E)

        dwe   = corrected_dwe(occ_logits, gt.unsqueeze(0).unsqueeze(0))
        prob  = torch.sigmoid(occ_logits.squeeze())
        pred  = (prob > THRESH).float()
        gt_b  = (gt > 0.5).float()

        tp = (pred * gt_b).sum().item()
        fp = (pred * (1 - gt_b)).sum().item()
        fn = ((1 - pred) * gt_b).sum().item()
        iou  = tp / (tp + fp + fn + 1e-6)
        prec = tp / (tp + fp + 1e-6)
        rec  = tp / (tp + fn + 1e-6)
        f1   = 2 * prec * rec / (prec + rec + 1e-6)
        gt_density = gt_b.mean().item() * 100

        iou_near, iou_far = near_far_iou(pred, gt_b)

        results.append({
            'idx': idx, 'iou': iou, 'dwe': dwe,
            'precision': prec, 'recall': rec, 'f1': f1,
            'gt_density': gt_density,
            'iou_near': iou_near, 'iou_far': iou_far,
            'prob': prob.cpu().numpy(),
            'pred': pred.cpu().numpy(),
            'gt':   gt_b.cpu().numpy()
        })
        if i % 10 == 0:
            print(f"  {i+1:3d}/81 | IoU {iou:.3f} | DWE {dwe:.4f}")

# Summary
miou   = np.mean([r['iou']       for r in results])
mdwe   = np.mean([r['dwe']       for r in results])
mprec  = np.mean([r['precision'] for r in results])
mrec   = np.mean([r['recall']    for r in results])
mf1    = np.mean([r['f1']        for r in results])
mnear  = np.mean([r['iou_near']  for r in results])
mfar   = np.mean([r['iou_far']   for r in results])

print("\n" + "═"*55)
print(f"  V3 FINAL EVALUATION")
print("═"*55)
print(f"  IoU        : {miou:.4f}   (V2: 0.3011  | Δ +{miou-0.3011:.4f})")
print(f"  DWE        : {mdwe:.4f}   (V2: 0.2323  | Δ {mdwe-0.2323:+.4f})")
print(f"  Precision  : {mprec:.4f}")
print(f"  Recall     : {mrec:.4f}")
print(f"  F1         : {mf1:.4f}")
print(f"  IoU Near   : {mnear:.4f}  (≤30 cells from ego)")
print(f"  IoU Far    : {mfar:.4f}  (>30 cells from ego)")
print("═"*55)

    1/81 | IoU 0.306 | DWE 0.1334
   11/81 | IoU 0.239 | DWE 0.1761
   21/81 | IoU 0.271 | DWE 0.2206
   31/81 | IoU 0.289 | DWE 0.1713
   41/81 | IoU 0.222 | DWE 0.2378
   51/81 | IoU 0.542 | DWE 0.0236
   61/81 | IoU 0.299 | DWE 0.1096
   71/81 | IoU 0.306 | DWE 0.1849
   81/81 | IoU 0.261 | DWE 0.0762

═══════════════════════════════════════════════════════
  V3 FINAL EVALUATION
═══════════════════════════════════════════════════════
  IoU        : 0.3649   (V2: 0.3011  | Δ +0.0638)
  DWE        : 0.1137   (V2: 0.2323  | Δ -0.1186)
  Precision  : 0.4520
  Recall     : 0.6110
  F1         : 0.5146
  IoU Near   : 0.6278  (≤30 cells from ego)
  IoU Far    : 0.3150  (>30 cells from ego)
═══════════════════════════════════════════════════════


In [ ]:
def plot_bev_sample(r, title, save_path):
    """3-panel: GT | Prediction | Confidence heatmap"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor('#0f172a')

    cmap_occ  = LinearSegmentedColormap.from_list('occ', ['#0f172a', '#38bdf8'])
    cmap_conf = LinearSegmentedColormap.from_list('conf', ['#0f172a', '#fbbf24', '#f87171'])

    # Ego circle helper
    def draw_ego(ax):
        for r_val, col, lw in [(30, '#ffffff', 0.8), (60, '#475569', 0.5)]:
            circle = plt.Circle((100, 100), r_val, color=col,
                                 fill=False, linewidth=lw, linestyle='--')
            ax.add_patch(circle)
        ax.plot(100, 100, 'w+', markersize=10, markeredgewidth=2)

    # Panel 1: Ground Truth
    axes[0].imshow(r['gt'], cmap=cmap_occ, vmin=0, vmax=1, origin='upper')
    draw_ego(axes[0])
    axes[0].set_title('Ground Truth', color='white', fontsize=13, pad=8)
    axes[0].set_facecolor('#0f172a')
    axes[0].axis('off')

    # Panel 2: Prediction (binary)
    axes[1].imshow(r['pred'], cmap=cmap_occ, vmin=0, vmax=1, origin='upper')
    draw_ego(axes[1])
    axes[1].set_title(f"Prediction  |  IoU {r['iou']:.3f}", color='white', fontsize=13, pad=8)
    axes[1].set_facecolor('#0f172a')
    axes[1].axis('off')

    # Panel 3: Confidence heatmap
    axes[2].imshow(r['prob'], cmap=cmap_conf, vmin=0, vmax=1, origin='upper')
    draw_ego(axes[2])
    axes[2].set_title(f"Confidence  |  DWE {r['dwe']:.4f}", color='white', fontsize=13, pad=8)
    axes[2].set_facecolor('#0f172a')
    axes[2].axis('off')

    fig.suptitle(title, color='white', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight',
                facecolor='#0f172a', edgecolor='none')
    plt.close()
    print(f"  Saved: {save_path}")

# Sort by IoU
sorted_r = sorted(results, key=lambda x: x['iou'])
best_5   = sorted_r[-5:][::-1]
worst_5  = sorted_r[:5]
import random
random_5 = random.sample(results, 5)

for i, r in enumerate(best_5):
    plot_bev_sample(r, f"Best #{i+1} | IoU {r['iou']:.3f} | GT density {r['gt_density']:.1f}%",
                    f"{VIZ_DIR}/best_{i+1}.png")

for i, r in enumerate(worst_5):
    plot_bev_sample(r, f"Worst #{i+1} | IoU {r['iou']:.3f} | GT density {r['gt_density']:.1f}%",
                    f"{VIZ_DIR}/worst_{i+1}.png")

for i, r in enumerate(random_5):
    plot_bev_sample(r, f"Random #{i+1} | IoU {r['iou']:.3f} | DWE {r['dwe']:.4f}",
                    f"{VIZ_DIR}/random_{i+1}.png")

  Saved: /content/eval_viz/best_1.png
  Saved: /content/eval_viz/best_2.png
  Saved: /content/eval_viz/best_3.png
  Saved: /content/eval_viz/best_4.png
  Saved: /content/eval_viz/best_5.png
  Saved: /content/eval_viz/worst_1.png
  Saved: /content/eval_viz/worst_2.png
  Saved: /content/eval_viz/worst_3.png
  Saved: /content/eval_viz/worst_4.png
  Saved: /content/eval_viz/worst_5.png
  Saved: /content/eval_viz/random_1.png
  Saved: /content/eval_viz/random_2.png
  Saved: /content/eval_viz/random_3.png
  Saved: /content/eval_viz/random_4.png
  Saved: /content/eval_viz/random_5.png


In [ ]:
# Paste your full training log here
epochs = list(range(1, 61))
val_iou = [0.1036,0.1293,0.1564,0.1826,0.1699,0.2035,0.1993,0.2243,0.2163,0.2484,
           0.2496,0.2548,0.2571,0.2412,0.2545,0.2563,0.2699,0.2794,0.2630,0.2690,
           0.2979,0.2851,0.2935,0.2972,0.3031,0.3114,0.3146,0.3105,0.3075,0.3091,
           0.3212,0.3263,0.3287,0.3338,0.3380,0.3387,0.3464,0.3475,0.3402,0.3470,
           0.3499,0.3540,0.3547,0.3575,0.3573,0.3572,0.3571,0.3593,0.3576,0.3604,
           0.3605,0.3627,0.3601,0.3634,0.3621,0.3633,0.3633,0.3608,0.3616,0.3649]
val_loss= [1.1853,1.1175,1.0646,1.0245,1.1715,1.1183,1.0727,1.0407,1.0122,0.9914,
           0.9705,0.9537,0.9358,0.9246,0.9024,0.8886,0.8776,0.8645,0.8495,0.8388,
           0.8270,0.8157,0.8035,0.7943,0.7817,0.7685,0.7610,0.7494,0.7398,0.7301,
           0.7192,0.7113,0.7007,0.6903,0.6791,0.6711,0.6612,0.6525,0.6457,0.6936,
           0.6862,0.6750,0.6645,0.6561,0.6480,0.6414,0.6335,0.6252,0.6210,0.6159,
           0.6104,0.6055,0.6016,0.5971,0.5975,0.5934,0.5926,0.5921,0.5910,0.5902]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
fig.patch.set_facecolor('#0f172a')
for ax in [ax1, ax2]:
    ax.set_facecolor('#1e293b')
    ax.tick_params(colors='#94a3b8')
    ax.spines['bottom'].set_color('#334155')
    ax.spines['left'].set_color('#334155')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Phase shading on both axes
for ax in [ax1, ax2]:
    ax.axvspan(0.5, 4.5,  alpha=0.12, color='#a78bfa')
    ax.axvspan(4.5, 39.5, alpha=0.08, color='#38bdf8')
    ax.axvspan(39.5,60.5, alpha=0.08, color='#34d399')

# IoU plot
ax1.plot(epochs, val_iou, color='#38bdf8', linewidth=2.5, label='Val IoU')
ax1.axhline(0.3011, color='#f87171', linestyle='--', linewidth=1.5, label='V2 baseline 0.3011')
ax1.axhline(0.3649, color='#34d399', linestyle=':', linewidth=1.5, label='V3 best 0.3649')
ax1.fill_between(epochs, val_iou, alpha=0.12, color='#38bdf8')
ax1.set_ylabel('Val IoU', color='#e2e8f0', fontsize=12)
ax1.legend(facecolor='#1e293b', labelcolor='white', fontsize=10, loc='upper left')
ax1.yaxis.label.set_color('#e2e8f0')

# Phase labels
for x, label, col in [(2.5,'Warmup','#a78bfa'), (22,'Phase 1','#38bdf8'), (50,'Phase 2','#34d399')]:
    ax1.text(x, ax1.get_ylim()[0] + 0.005, label, color=col, fontsize=10, ha='center')

# Loss plot
ax2.plot(epochs, val_loss, color='#fb923c', linewidth=2.5, label='Val Loss')
ax2.fill_between(epochs, val_loss, alpha=0.12, color='#fb923c')
ax2.set_ylabel('Val Loss', color='#e2e8f0', fontsize=12)
ax2.set_xlabel('Epoch',    color='#e2e8f0', fontsize=12)
ax2.legend(facecolor='#1e293b', labelcolor='white', fontsize=10)
ax2.yaxis.label.set_color('#e2e8f0')

fig.suptitle('V3 Training Curves — IoU & Loss over 60 Epochs',
             color='white', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/training_curves.png', dpi=150, bbox_inches='tight',
            facecolor='#0f172a')
plt.close()
print("Training curves saved.")

Training curves saved.


In [ ]:
ious = [r['iou']  for r in results]
dwes = [r['dwe']  for r in results]
dens = [r['gt_density'] for r in results]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#0f172a')

for ax in axes:
    ax.set_facecolor('#1e293b')
    ax.tick_params(colors='#94a3b8')
    for spine in ax.spines.values():
        spine.set_color('#334155')

# IoU distribution
axes[0].hist(ious, bins=20, color='#38bdf8', alpha=0.85, edgecolor='#0f172a')
axes[0].axvline(np.mean(ious), color='#fbbf24', linewidth=2, label=f'Mean {np.mean(ious):.3f}')
axes[0].axvline(0.3011, color='#f87171', linewidth=2, linestyle='--', label='V2 0.3011')
axes[0].set_xlabel('IoU', color='#e2e8f0'); axes[0].set_ylabel('Count', color='#e2e8f0')
axes[0].set_title('IoU Distribution', color='white', fontsize=13)
axes[0].legend(facecolor='#1e293b', labelcolor='white', fontsize=10)

# DWE distribution
axes[1].hist(dwes, bins=20, color='#34d399', alpha=0.85, edgecolor='#0f172a')
axes[1].axvline(np.mean(dwes), color='#fbbf24', linewidth=2, label=f'Mean {np.mean(dwes):.3f}')
axes[1].axvline(0.2323, color='#f87171', linewidth=2, linestyle='--', label='V2 0.2323')
axes[1].set_xlabel('DWE', color='#e2e8f0'); axes[1].set_ylabel('Count', color='#e2e8f0')
axes[1].set_title('DWE Distribution', color='white', fontsize=13)
axes[1].legend(facecolor='#1e293b', labelcolor='white', fontsize=10)

# IoU vs GT Density scatter
sc = axes[2].scatter(dens, ious, c=dwes, cmap='RdYlGn_r', s=50, alpha=0.85, edgecolors='none')
plt.colorbar(sc, ax=axes[2], label='DWE').ax.yaxis.label.set_color('white')
axes[2].set_xlabel('GT Density %', color='#e2e8f0')
axes[2].set_ylabel('IoU',          color='#e2e8f0')
axes[2].set_title('IoU vs Scene Density\n(color = DWE)', color='white', fontsize=13)

plt.suptitle('Per-Sample Metric Distributions (81 Val Samples)',
             color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/metric_distributions.png', dpi=150, bbox_inches='tight',
            facecolor='#0f172a')
plt.close()
print("Distributions saved.")

Distributions saved.


In [ ]:
# V2 vs V3 comparison bar chart
metrics   = ['IoU', 'Precision', 'Recall', 'F1']
v2_vals   = [0.3011, 0.4369, 0.5218, 0.4734]   # from your notebook
v3_vals   = [miou,   mprec,  mrec,   mf1]

x = np.arange(len(metrics))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#1e293b')
ax.tick_params(colors='#94a3b8')
for spine in ax.spines.values(): spine.set_color('#334155')

bars_v2 = ax.bar(x - w/2, v2_vals, w, label='V2', color='#f87171', alpha=0.85)
bars_v3 = ax.bar(x + w/2, v3_vals, w, label='V3', color='#38bdf8', alpha=0.85)

for bar in bars_v2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom',
            color='white', fontsize=10)
for bar in bars_v3:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom',
            color='white', fontsize=10)

ax.set_xticks(x); ax.set_xticklabels(metrics, color='#e2e8f0', fontsize=12)
ax.set_ylabel('Score', color='#e2e8f0', fontsize=12)
ax.set_ylim(0, 0.55)
ax.set_title('V2 vs V3 — All Metrics Comparison', color='white', fontsize=14, pad=12)
ax.legend(facecolor='#1e293b', labelcolor='white', fontsize=11)
plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/v2_vs_v3_comparison.png', dpi=150, bbox_inches='tight',
            facecolor='#0f172a')
plt.close()

# Save full JSON report
report = {
    'model': 'BEVOccupancyModel V3.1',
    'checkpoint': f'best_iou_model.pth (epoch {ckpt["epoch"]})',
    'val_samples': 81,
    'metrics': {
        'iou': round(miou, 4),
        'dwe_corrected': round(mdwe, 4),
        'precision': round(mprec, 4),
        'recall': round(mrec, 4),
        'f1': round(mf1, 4),
        'iou_near_ego': round(mnear, 4),
        'iou_far_field': round(mfar, 4),
    },
    'vs_v2': {
        'delta_iou': round(miou - 0.3011, 4),
        'delta_dwe': round(mdwe - 0.2323, 4),
        'iou_improvement_pct': round((miou - 0.3011) / 0.3011 * 100, 1),
        'dwe_improvement_pct': round((0.2323 - mdwe) / 0.2323 * 100, 1),
    }
}
with open(f'{VIZ_DIR}/eval_report_v3.json', 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print(f"\nAll outputs saved to: {VIZ_DIR}")

{
  "model": "BEVOccupancyModel V3.1",
  "checkpoint": "best_iou_model.pth (epoch 60)",
  "val_samples": 81,
  "metrics": {
    "iou": 0.3649,
    "dwe_corrected": 0.1137,
    "precision": 0.452,
    "recall": 0.611,
    "f1": 0.5146,
    "iou_near_ego": 0.6278,
    "iou_far_field": 0.315
  },
  "vs_v2": {
    "delta_iou": 0.0638,
    "delta_dwe": -0.1186,
    "iou_improvement_pct": 21.2,
    "dwe_improvement_pct": 51.1
  }
}

All outputs saved to: /content/eval_viz
